<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px"> 

#  APIs for financial data - Solutions

---

<br>
For both parts of this lab we will use Python to interact with the Alpha Vantage API and export data from different endpoints to answer two research questions.

# Part 1: GDP and Consumer Sentiment during the pandemic

You have been tasked with analysing how various economic indicators have behaved during the last few years. Your team has already identified a free source of this data: Alpha Vantage (https://www.alphavantage.co). It is now up to you to extract the relevant data about these indicators and write a short report about your findings.

***Note: be sure to sign up for a free API key at https://www.alphavantage.co/support/#api-key (more detailed instructions are in the instruction document for this lab)***

**0. Imports**
Put all Python imports in the cell below. If you later decide you need to import something, you must put it here and re-run!

In [42]:
import requests
import pandas as pd
import plotly.express as px
import time

#Refresh Jupyter cache. Prevents use of old or previous imports / caches.
import importlib
import config

importlib.reload(config)

from pathlib import Path

# Account Key is stored in a config file
from config import ALPHAVANTAGE_API_KEY

# Create Output Folder
output_folder = Path("API_Output_CSV")
output_folder.mkdir(exist_ok=True)

**1. Find the correct API endpoint to retrieve historical data on GDP**

Define a variable `base_url` that is a **string** that is simply the base portion of the Alpha Vantage URL endpoint. That is, if you look at any API call, this is everything before the question mark.

Next, define another variable `API_KEY` that is a **string** and is your Alpha Vantage API key (make one, they're free and you don't need to give a real email!)

Use the [Documentation](https://www.alphavantage.co/documentation/) to help you.

In [43]:
# STEP 1. DEFINE URL + PARAMETERS

# API endpoint
base_url = 'https://www.alphavantage.co/query'

# Parameters
params = {
    'function': 'REAL_GDP',
    'interval': 'quarterly',
    'datatype': 'json',             #returns the time series in JSON format
    'apikey': ALPHAVANTAGE_API_KEY
}

**2. Use Python and the `requests` library to make an API call and retrieve historical GDP data at *the highest level of granularity* (i.e. most frequent that the API allows). Remember to add your own API key to the query.**

You should convert the results to JSON so it behaves like a Python dictionary.

In [44]:
# STEP 2. API REQUEST + ERROR HANDLING

# 1. Calls the API via requests and safely handles possible errors

try:
    # Send GET request to API with headers and parameters
    response = requests.get(base_url, params=params)

    # Pause briefly to avoid hitting API rate limits
    time.sleep(1)

    # Raise an error if the response status is not successful
    response.raise_for_status()

    # Parse JSON response and convert to Python dictionary
    real_gdp_quarterly = response.json()

# 2. Handles HTTP errors 
# Catches API-related errors (e.g., 401, 404, 408, 429)

except requests.exceptions.HTTPError as e:
    print("HTTP error:", response.status_code, response.text)
    real_gdp_quarterly = [] # reset to empty dataset

# 3. Handles other errors 
# Catches unexpected errors (e.g., connection issues, parsing errors)

except Exception as e:
    print("Other error:", e)
    real_gdp_quarterly = [] # reset to empty dataset

In [45]:
real_gdp_quarterly

{'name': 'Real Gross Domestic Product',
 'interval': 'quarterly',
 'unit': 'billions of dollars',
 'data': [{'date': '2026-01-01', 'value': '5900.814'},
  {'date': '2025-10-01', 'value': '6119.385'},
  {'date': '2025-07-01', 'value': '6023.785'},
  {'date': '2025-04-01', 'value': '5943.384'},
  {'date': '2025-01-01', 'value': '5776.724'},
  {'date': '2024-10-01', 'value': '5997.184'},
  {'date': '2024-07-01', 'value': '5884.567'},
  {'date': '2024-04-01', 'value': '5829.384'},
  {'date': '2024-01-01', 'value': '5647.3'},
  {'date': '2023-10-01', 'value': '5847.063'},
  {'date': '2023-07-01', 'value': '5731.929'},
  {'date': '2023-04-01', 'value': '5655.781'},
  {'date': '2023-01-01', 'value': '5488.947'},
  {'date': '2022-10-01', 'value': '5670.022'},
  {'date': '2022-07-01', 'value': '5556.056'},
  {'date': '2022-04-01', 'value': '5495.481'},
  {'date': '2022-01-01', 'value': '5354.373'},
  {'date': '2021-10-01', 'value': '5600.549'},
  {'date': '2021-07-01', 'value': '5418.754'},
  {

**3. Convert the data in this JSON to a Pandas dataframe and export to a csv.**

In [46]:
# STEP 3. CONVERT DATA IN JSON TO PANDAS DATAFRAME + DATA WRANGLING + COMPUTE DERIVED FIELDS + EXPORT TO A CSV FILE

# 1. Extract the 'data' portion and convert to a DataFrame

real_gdp_quarterly_df = pd.DataFrame(real_gdp_quarterly['data'])
real_gdp_quarterly_df

# 2. Convert data types

real_gdp_quarterly_df['date'] = pd.to_datetime(real_gdp_quarterly_df['date'])
real_gdp_quarterly_df['value'] = real_gdp_quarterly_df['value'].astype(float)

# 3. Sort date by ascending order and resets the order of the index 
# This is important for time series analysis

real_gdp_quarterly_df = real_gdp_quarterly_df.sort_values('date').reset_index(drop=True)

# 4. Compute the Derived fields

    # A. Quarterly Change
    # Measures the absolute change from previous quarter
    # Useful for identifying acceleration/slowing growth and recession analysis

real_gdp_quarterly_df['quarterly_change'] = (
    real_gdp_quarterly_df['value'].diff()
)

    # B. Quarterly Growth %
    # Useful for comparing economic growth periods and comparing GDP trends against VTI/VXX

real_gdp_quarterly_df['quarterly_growth_pct'] = (
    real_gdp_quarterly_df['value'].pct_change()
) * 100

    # C. Cumulative GDP Growth %
    # Good for long-term comparison
    # Useful for showing overall economic expansion

real_gdp_quarterly_df['cumulative_growth_pct'] = (
    (
        real_gdp_quarterly_df['value']
        / real_gdp_quarterly_df['value'].iloc[0]
    ) - 1
) * 100

    # D. 4-Quarter Moving Average
    # Smooths annual trend 

real_gdp_quarterly_df['moving_avg_4q'] = (
    real_gdp_quarterly_df['value']
    .rolling(4)
    .mean()
)

    # E. 8-Quarter Moving Average
    # Longer economic cycle trend

real_gdp_quarterly_df['moving_avg_8q'] = (
    real_gdp_quarterly_df['value']
    .rolling(8)
    .mean()
)

# 5. Add info from the Metadata

real_gdp_quarterly_df['name'] = real_gdp_quarterly['name']
real_gdp_quarterly_df['interval'] = real_gdp_quarterly['interval']
real_gdp_quarterly_df['unit'] = real_gdp_quarterly['unit']

# 6. Round numeric columns to 2 decimal places

numeric_columns = real_gdp_quarterly_df.select_dtypes(include='number').columns

real_gdp_quarterly_df[numeric_columns] = (
    real_gdp_quarterly_df[numeric_columns]
    .round(2)
)

# 7. View the data

real_gdp_quarterly_df

,date,value,quarterly_change,quarterly_growth_pct,cumulative_growth_pct,moving_avg_4q,moving_avg_8q,name,interval,unit
0,2002-01-01,3501.12,NaN,NaN,0.00,NaN,NaN,Real Gross Domestic Product,quarterly,billions of dollars
1,2002-04-01,3608.50,107.38,3.07,3.07,NaN,NaN,Real Gross Domestic Product,quarterly,billions of dollars
2,2002-07-01,3650.25,41.76,1.16,4.26,NaN,NaN,Real Gross Domestic Product,quarterly,billions of dollars
3,2002-10-01,3712.84,62.59,1.71,6.05,3618.18,NaN,Real Gross Domestic Product,quarterly,billions of dollars
4,2003-01-01,3582.77,-130.08,-3.50,2.33,3638.59,NaN,Real Gross Domestic Product,quarterly,billions of dollars
...,...,...,...,...,...,...,...,...,...,...
92,2025-01-01,5776.72,-220.46,-3.68,65.00,5871.96,5796.24,Real Gross Domestic Product,quarterly,billions of dollars
93,2025-04-01,5943.38,166.66,2.89,69.76,5900.46,5832.19,Real Gross Domestic Product,quarterly,billions of dollars
94,2025-07-01,6023.78,80.40,1.35,72.05,5935.27,5868.67,Real Gross Domestic Product,quarterly,billions of dollars
95,2025-10-01,6119.38,95.60,1.59,74.78,5965.82,5902.71,Real Gross Domestic Product,quarterly,billions of dollars


In [47]:
# 8. View dataframe: Check datatypes & count of rows
real_gdp_quarterly_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   97 non-null     datetime64[us]
 1   value                  97 non-null     float64       
 2   quarterly_change       96 non-null     float64       
 3   quarterly_growth_pct   96 non-null     float64       
 4   cumulative_growth_pct  97 non-null     float64       
 5   moving_avg_4q          94 non-null     float64       
 6   moving_avg_8q          90 non-null     float64       
 7   name                   97 non-null     str           
 8   interval               97 non-null     str           
 9   unit                   97 non-null     str           
dtypes: datetime64[us](1), float64(6), str(3)
memory usage: 7.7 KB


In [48]:
# 9. Re-order columns
real_gdp_quarterly_df = real_gdp_quarterly_df[
    [
        'date',
        'name',
        'interval',
        'unit',
        'value',
        'quarterly_change',
        'quarterly_growth_pct',
        'cumulative_growth_pct',
        'moving_avg_4q',
        'moving_avg_8q'
    ]
]

real_gdp_quarterly_df

,date,name,interval,unit,value,quarterly_change,quarterly_growth_pct,cumulative_growth_pct,moving_avg_4q,moving_avg_8q
0,2002-01-01,Real Gross Domestic Product,quarterly,billions of dollars,3501.12,NaN,NaN,0.00,NaN,NaN
1,2002-04-01,Real Gross Domestic Product,quarterly,billions of dollars,3608.50,107.38,3.07,3.07,NaN,NaN
2,2002-07-01,Real Gross Domestic Product,quarterly,billions of dollars,3650.25,41.76,1.16,4.26,NaN,NaN
3,2002-10-01,Real Gross Domestic Product,quarterly,billions of dollars,3712.84,62.59,1.71,6.05,3618.18,NaN
4,2003-01-01,Real Gross Domestic Product,quarterly,billions of dollars,3582.77,-130.08,-3.50,2.33,3638.59,NaN
...,...,...,...,...,...,...,...,...,...,...
92,2025-01-01,Real Gross Domestic Product,quarterly,billions of dollars,5776.72,-220.46,-3.68,65.00,5871.96,5796.24
93,2025-04-01,Real Gross Domestic Product,quarterly,billions of dollars,5943.38,166.66,2.89,69.76,5900.46,5832.19
94,2025-07-01,Real Gross Domestic Product,quarterly,billions of dollars,6023.78,80.40,1.35,72.05,5935.27,5868.67
95,2025-10-01,Real Gross Domestic Product,quarterly,billions of dollars,6119.38,95.60,1.59,74.78,5965.82,5902.71


In [49]:
# 10. Export to CSV and save inside output folder
real_gdp_quarterly_df.to_csv(
    output_folder / "real_gdp_quarterly.csv",
    index=False
)

**4. Next, identify the endpoint for finding daily stock price values and query the VXX ticker.**

Make sure to get as much data as possible using the `outputsize` parameter at this endpoint.

VXX is a mutual fund that that adequately represents the [VIX](https://www.investopedia.com/articles/optioninvestor/09/implied-volatility-contrary-indicator.asp#:~:text=VIX%20measures%20the%20market%27s%20expectation%20of%20volatility%20over,trends%20in%20the%20VIX%20can%20inform%20trading%20strategies.) index that represents the fear and volatility in the market. When VIX (or VXX) is high, fear controls the market, and when VIX (or VXX) is low, people have more confidence in the market.

#### <font color='green'>**The Volatility Index: Reading Market Sentiment**</font><br>

Known ominously among investors as the "fear index" and launched by the Chicago Board Options Exchange (now the Cboe) in 1993, <br>
the Volatility Index (VIX) is meant to present the market's expectation of volatility over the coming 30 days.<br> 
The metric is derived from options prices on the S&P 500 Index and captures the anticipated swings that drive investor sentiment.<br>

A mantra investors learn early on is, "When the VIX is high, it's time to buy. When the VIX is low, look out below!"<br>

**Why is the VIX Sometimes Referred to as the "Fear Gauge"?**<br>
The VIX is sometimes called the "fear gauge" because it reflects market participants' anxiety about future market downturns.

**The Bottom Line**<br>
VIX measures the market's expectation of volatility over the next 30 days based on S&P 500 index options.<br> 
A higher VIX value indicates greater anticipated volatility and market uncertainty, while a lower VIX value suggests market stability.<br>
Key levels and trends in the VIX can inform trading strategies.

In [80]:
# STEP 1. DEFINE URL AND PARAMETERS

# API endpoint
base_url = 'https://www.alphavantage.co/query'

# Parameters
params = {
    'function': 'TIME_SERIES_WEEKLY',    # returns raw (as-traded) daily time series of the global equity specified
    'symbol': 'VXX',                    # the global equity specified: VXX is a mutual fund that that adequately represents the VIX
    'datatype': 'json',          # returns the daily time series in JSON format
    'apikey': ALPHAVANTAGE_API_KEY
}

In [81]:
# STEP 2. API REQUEST + ERROR HANDLING

# 1. Calls the API via requests and safely handles possible errors

try:
    # Send GET request to API with headers and parameters
    response = requests.get(base_url, params=params)

    # Pause briefly to avoid hitting API rate limits
    time.sleep(1)

    # Raise an error if the response status is not successful
    response.raise_for_status()

    # Parse JSON response and convert to Python dictionary
    time_series_weekly_vxx = response.json()

# 2. Handles HTTP errors 
# Catches API-related errors (e.g., 401, 404, 408, 429)

except requests.exceptions.HTTPError as e:
    print("HTTP error:", response.status_code, response.text)
    time_series_weekly_vxx = [] # reset to empty dataset

# 3. Handles other errors 
# Catches unexpected errors (e.g., connection issues, parsing errors)

except Exception as e:
    print("Other error:", e)
    time_series_weekly_vxx = [] # reset to empty dataset

**5. Cast the result as a dataframe and export it to a csv.**

In [82]:
time_series_weekly_vxx

{'Meta Data': {'1. Information': 'Weekly Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'VXX',
  '3. Last Refreshed': '2026-05-08',
  '4. Time Zone': 'US/Eastern'},
 'Weekly Time Series': {'2026-05-08': {'1. open': '28.3500',
   '2. high': '29.2400',
   '3. low': '27.5600',
   '4. close': '28.0600',
   '5. volume': '38682038'},
  '2026-05-01': {'1. open': '29.4800',
   '2. high': '29.6300',
   '3. low': '27.7700',
   '4. close': '28.4000',
   '5. volume': '29256666'},
  '2026-04-24': {'1. open': '29.2900',
   '2. high': '30.9450',
   '3. low': '29.0700',
   '4. close': '29.7900',
   '5. volume': '39941119'},
  '2026-04-17': {'1. open': '31.1600',
   '2. high': '31.3000',
   '3. low': '28.3700',
   '4. close': '28.9800',
   '5. volume': '35726903'},
  '2026-04-10': {'1. open': '34.8000',
   '2. high': '36.3300',
   '3. low': '30.0950',
   '4. close': '30.7700',
   '5. volume': '49587252'},
  '2026-04-02': {'1. open': '37.9100',
   '2. high': '39.9288',
   '3. low': '34.210

In [94]:
# STEP 3. CONVERT DATA IN JSON TO PANDAS DATAFRAME + DATA WRANGLING + COMPUTE DERIVED FIELDS + EXPORT TO A CSV FILE

# 1. Extract the 'Weekly Prices (open, high, low, close) and Volumes' portion and convert to a DataFrame
time_series_weekly_vxx_df = pd.DataFrame.from_dict(
    time_series_weekly_vxx['Weekly Time Series'],
    orient='index'
)

# 2. Reset index so dates become a column instead of an index
time_series_weekly_vxx_df.reset_index(inplace=True)

# 3. Rename columns
time_series_weekly_vxx_df.columns = [
    'date',
    'open',
    'high',
    'low',
    'close',
    'volume'
]

# 4. Convert data types
time_series_weekly_vxx_df['date'] = pd.to_datetime(time_series_weekly_vxx_df['date'])

numeric_columns = ['open', 'high', 'low', 'close', 'volume']

time_series_weekly_vxx_df[numeric_columns] = (
    time_series_weekly_vxx_df[numeric_columns].astype(float)
)

time_series_weekly_vxx_df['volume'] = time_series_weekly_vxx_df['volume'].astype(int)

# 5. Sort date by ascending order and resets the order of the index
# This is important for time series analysis

time_series_weekly_vxx_df = time_series_weekly_vxx_df.sort_values('date').reset_index(drop=True)

# 6. Compute the Derived fields

    # A. Daily Price Movement
time_series_weekly_vxx_df['daily_change'] = (
    time_series_weekly_vxx_df['close'] - time_series_weekly_vxx_df['open']
)

    # B. Intraday Return %
time_series_weekly_vxx_df['intraday_return_pct'] = (
    (time_series_weekly_vxx_df['close'] - time_series_weekly_vxx_df['open'])
    / time_series_weekly_vxx_df['open']
) * 100

    # C. Day-over-day return
time_series_weekly_vxx_df['daily_return_pct'] = (
    time_series_weekly_vxx_df['close']
    .pct_change()
) * 100

    # D. Cumulative return %
    # iloc[0] refers to the initial close
time_series_weekly_vxx_df["cumulative_return_pct"] = (
    (time_series_weekly_vxx_df["close"] / time_series_weekly_vxx_df["close"].iloc[0]) - 1
) * 100

    # E. Intraday Volatility
time_series_weekly_vxx_df['intraday_volatility_abs'] = (
    time_series_weekly_vxx_df['high'] - time_series_weekly_vxx_df['low']
)

    # F. Volatility Ratio
time_series_weekly_vxx_df['volatility_ratio'] = (
    time_series_weekly_vxx_df['intraday_volatility_abs']
    / time_series_weekly_vxx_df['close']
)

    # G. Intraday Volatility %
time_series_weekly_vxx_df['intraday_volatility_pct'] = (
    (
        time_series_weekly_vxx_df['high']
        - time_series_weekly_vxx_df['low']
    )
    / time_series_weekly_vxx_df['open']
) * 100

    # H. Volume Change %
time_series_weekly_vxx_df['volume_change_pct'] = (
    time_series_weekly_vxx_df['volume']
    .pct_change()
) * 100

    # I. 7-Day Moving Average
time_series_weekly_vxx_df['moving_avg_7d'] = (
    time_series_weekly_vxx_df['close']
    .rolling(7)
    .mean()
)

    # J. 30-Day Moving Average
time_series_weekly_vxx_df['moving_avg_30d'] = (
    time_series_weekly_vxx_df['close']
    .rolling(30)
    .mean()
)

# 7. Add Symbol and Time Zone from the Metadata

time_series_weekly_vxx_df['symbol'] = time_series_weekly_vxx['Meta Data']['2. Symbol']
time_series_weekly_vxx_df['dt_last_refreshed'] = time_series_weekly_vxx['Meta Data']['3. Last Refreshed']
time_series_weekly_vxx_df['timezone'] = time_series_weekly_vxx['Meta Data']['4. Time Zone']

# 8. Convert data types from string to date
time_series_weekly_vxx_df['dt_last_refreshed'] = pd.to_datetime(time_series_weekly_vxx_df['dt_last_refreshed'])

# 8. Round numeric columns to 2 decimal places
numeric_columns = time_series_weekly_vxx_df.select_dtypes(include='number').columns

time_series_weekly_vxx_df[numeric_columns] = (
    time_series_weekly_vxx_df[numeric_columns]
    .round(2)
)

# 9. View the data
time_series_weekly_vxx_df

,date,open,high,low,close,volume,daily_change,intraday_return_pct,daily_return_pct,cumulative_return_pct,intraday_volatility_abs,volatility_ratio,intraday_volatility_pct,volume_change_pct,moving_avg_7d,moving_avg_30d,symbol,dt_last_refreshed,timezone
0,2009-02-06,108.10,108.10,95.06,97.70,1024000,-10.40,-9.62,NaN,0.00,13.04,0.13,12.06,NaN,NaN,NaN,VXX,2026-05-08,US/Eastern
1,2009-02-13,98.33,108.00,97.87,101.99,665600,3.66,3.72,4.39,4.39,10.13,0.10,10.30,-35.00,NaN,NaN,VXX,2026-05-08,US/Eastern
2,2009-02-20,107.00,116.40,105.00,113.32,358400,6.32,5.91,11.11,15.99,11.40,0.10,10.65,-46.15,NaN,NaN,VXX,2026-05-08,US/Eastern
3,2009-02-27,111.00,120.00,102.96,108.67,569600,-2.33,-2.10,-4.10,11.23,17.04,0.16,15.35,58.93,NaN,NaN,VXX,2026-05-08,US/Eastern
4,2009-03-06,111.02,118.55,104.79,113.38,870400,2.36,2.13,4.33,16.05,13.76,0.12,12.39,52.81,NaN,NaN,VXX,2026-05-08,US/Eastern
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
896,2026-04-10,34.80,36.33,30.10,30.77,49587252,-4.03,-11.58,-11.33,-68.51,6.24,0.20,17.92,-20.55,34.36,31.76,VXX,2026-05-08,US/Eastern
897,2026-04-17,31.16,31.30,28.37,28.98,35726903,-2.18,-7.00,-5.82,-70.34,2.93,0.10,9.40,-27.95,34.37,31.62,VXX,2026-05-08,US/Eastern
898,2026-04-24,29.29,30.94,29.07,29.79,39941119,0.50,1.71,2.80,-69.51,1.88,0.06,6.40,11.80,33.53,31.52,VXX,2026-05-08,US/Eastern
899,2026-05-01,29.48,29.63,27.77,28.40,29256666,-1.08,-3.66,-4.67,-70.93,1.86,0.07,6.31,-26.75,32.57,31.35,VXX,2026-05-08,US/Eastern


In [95]:
# 10. View dataframe: Check datatypes & count of rows
time_series_weekly_vxx_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 901 entries, 0 to 900
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     901 non-null    datetime64[us]
 1   open                     901 non-null    float64       
 2   high                     901 non-null    float64       
 3   low                      901 non-null    float64       
 4   close                    901 non-null    float64       
 5   volume                   901 non-null    int64         
 6   daily_change             901 non-null    float64       
 7   intraday_return_pct      901 non-null    float64       
 8   daily_return_pct         900 non-null    float64       
 9   cumulative_return_pct    901 non-null    float64       
 10  intraday_volatility_abs  901 non-null    float64       
 11  volatility_ratio         901 non-null    float64       
 12  intraday_volatility_pct  901 non-null    float6

In [96]:
# 11. Re-order columns

time_series_weekly_vxx_df = time_series_weekly_vxx_df[
    [
        'date',
        'symbol',
        'dt_last_refreshed',
        'timezone',
        'open',
        'high',
        'low',
        'close',
        'volume',
        'daily_change',
        'intraday_return_pct',
        'daily_return_pct',
        'cumulative_return_pct',
        'intraday_volatility_abs',
        'volatility_ratio',
        'intraday_volatility_pct',
        'volume_change_pct',
        'moving_avg_7d',
        'moving_avg_30d'
    ]
]

time_series_weekly_vxx_df

,date,symbol,dt_last_refreshed,timezone,open,high,low,close,volume,daily_change,intraday_return_pct,daily_return_pct,cumulative_return_pct,intraday_volatility_abs,volatility_ratio,intraday_volatility_pct,volume_change_pct,moving_avg_7d,moving_avg_30d
0,2009-02-06,VXX,2026-05-08,US/Eastern,108.10,108.10,95.06,97.70,1024000,-10.40,-9.62,NaN,0.00,13.04,0.13,12.06,NaN,NaN,NaN
1,2009-02-13,VXX,2026-05-08,US/Eastern,98.33,108.00,97.87,101.99,665600,3.66,3.72,4.39,4.39,10.13,0.10,10.30,-35.00,NaN,NaN
2,2009-02-20,VXX,2026-05-08,US/Eastern,107.00,116.40,105.00,113.32,358400,6.32,5.91,11.11,15.99,11.40,0.10,10.65,-46.15,NaN,NaN
3,2009-02-27,VXX,2026-05-08,US/Eastern,111.00,120.00,102.96,108.67,569600,-2.33,-2.10,-4.10,11.23,17.04,0.16,15.35,58.93,NaN,NaN
4,2009-03-06,VXX,2026-05-08,US/Eastern,111.02,118.55,104.79,113.38,870400,2.36,2.13,4.33,16.05,13.76,0.12,12.39,52.81,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
896,2026-04-10,VXX,2026-05-08,US/Eastern,34.80,36.33,30.10,30.77,49587252,-4.03,-11.58,-11.33,-68.51,6.24,0.20,17.92,-20.55,34.36,31.76
897,2026-04-17,VXX,2026-05-08,US/Eastern,31.16,31.30,28.37,28.98,35726903,-2.18,-7.00,-5.82,-70.34,2.93,0.10,9.40,-27.95,34.37,31.62
898,2026-04-24,VXX,2026-05-08,US/Eastern,29.29,30.94,29.07,29.79,39941119,0.50,1.71,2.80,-69.51,1.88,0.06,6.40,11.80,33.53,31.52
899,2026-05-01,VXX,2026-05-08,US/Eastern,29.48,29.63,27.77,28.40,29256666,-1.08,-3.66,-4.67,-70.93,1.86,0.07,6.31,-26.75,32.57,31.35


In [97]:
# 12. Export to CSV and save inside output folder
time_series_weekly_vxx_df.to_csv(
    output_folder / "time_series_weekly_vxx_df.csv",
    index=False
)

**6. Repeat steps 4 and 5, but this time for VTI**

Use all of the same settings to now gather data for VTI, a mutual fund that consists of _every_ US stock. VTI can be considered an indicator for the overall market.

Again, as in step 5, save this data out to a csv.

#### <font color='green'>**VTI is the ticker symbol for the Vanguard Total Stock Market ETF**</font><br>

- It is a popular, low-cost exchange-traded fund designed to track the performance of the entire U.S. equity market, including large, mid, and small-cap stocks. 
- As of May 8, 2026, it trades around \(\$362-\$363\) with a very low expense ratio of 0.03%.

**Key Details (as of May 2026):**<br>
* Exchange: NYSE Arca
* Asset Class: Broad U.S. Equity
* Expense Ratio: 0.03% (approx. \(\$3\) annually per \(\$10,000\) invested)
* Top Holdings: Heavily weighted in major tech and growth companies like Apple, Microsoft, Nvidia, Amazon, and Meta.
* Purpose: Provides diversified exposure to ~3,700+ U.S. companies in a single fund.

VTI is often compared to the S&P 500 (VOO), with VTI offering broader exposure to smaller companies, whereas VOO focuses on ~500 large-cap, established companies.

In [98]:
# STEP 1. DEFINE URL AND PARAMETERS

# API endpoint
base_url = 'https://www.alphavantage.co/query'

# Parameters
params = {
    'function': 'TIME_SERIES_WEEKLY',    # returns raw (as-traded) daily time series of the global equity specified
    'symbol': 'VTI',             # the global equity specified: Vanguard Total Stock Market ETF
    'datatype': 'json',          # returns the daily time series in JSON format
    'apikey': ALPHAVANTAGE_API_KEY
}

In [99]:
# STEP 2. API REQUEST + ERROR HANDLING

# 1. Calls the API via requests and safely handles possible errors

try:
    # Send GET request to API with headers and parameters
    response = requests.get(base_url, params=params)

    # Pause briefly to avoid hitting API rate limits
    time.sleep(1)

    # Raise an error if the response status is not successful
    response.raise_for_status()

    # Parse JSON response and convert to Python dictionary
    time_series_weekly_vti = response.json()

# 2. Handles HTTP errors 
# Catches API-related errors (e.g., 401, 404, 408, 429)

except requests.exceptions.HTTPError as e:
    print("HTTP error:", response.status_code, response.text)
    time_series_weekly_vti = [] # reset to empty dataset

# 3. Handles other errors 
# Catches unexpected errors (e.g., connection issues, parsing errors)

except Exception as e:
    print("Other error:", e)
    time_series_weekly_vti = [] # reset to empty dataset

In [101]:
time_series_weekly_vti

{'Meta Data': {'1. Information': 'Weekly Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'VTI',
  '3. Last Refreshed': '2026-05-08',
  '4. Time Zone': 'US/Eastern'},
 'Weekly Time Series': {'2026-05-08': {'1. open': '355.0800',
   '2. high': '363.1250',
   '3. low': '352.5300',
   '4. close': '362.8700',
   '5. volume': '15897275'},
  '2026-05-01': {'1. open': '352.0000',
   '2. high': '357.1500',
   '3. low': '348.6700',
   '4. close': '355.2900',
   '5. volume': '15103707'},
  '2026-04-24': {'1. open': '349.9000',
   '2. high': '352.3700',
   '3. low': '346.5500',
   '4. close': '352.0500',
   '5. volume': '17522660'},
  '2026-04-17': {'1. open': '333.9400',
   '2. high': '351.7450',
   '3. low': '333.6400',
   '4. close': '350.5300',
   '5. volume': '18482054'},
  '2026-04-10': {'1. open': '323.9500',
   '2. high': '336.5500',
   '3. low': '321.4650',
   '4. close': '335.0500',
   '5. volume': '19371807'},
  '2026-04-02': {'1. open': '315.9000',
   '2. high': '325.0900'

In [102]:
# STEP 3. CONVERT DATA IN JSON TO PANDAS DATAFRAME + DATA WRANGLING + COMPUTE DERIVED FIELDS + EXPORT TO A CSV FILE

# 1. Extract the 'Weekly Prices (open, high, low, close) and Volumes' portion and convert to a DataFrame
time_series_weekly_vti_df = pd.DataFrame.from_dict(
    time_series_weekly_vti['Weekly Time Series'],
    orient='index'
)

# 2. Reset index so dates become a column instead of an index
time_series_weekly_vti_df.reset_index(inplace=True)

# 3. Rename columns
time_series_weekly_vti_df.columns = [
    'date',
    'open',
    'high',
    'low',
    'close',
    'volume'
]

# 4. Convert data types
time_series_weekly_vti_df['date'] = pd.to_datetime(time_series_weekly_vti_df['date'])

numeric_columns = ['open', 'high', 'low', 'close', 'volume']

time_series_weekly_vti_df[numeric_columns] = (
    time_series_weekly_vti_df[numeric_columns].astype(float)
)

time_series_weekly_vti_df['volume'] = time_series_weekly_vti_df['volume'].astype(int)

# 5. Sort date by ascending order and resets the order of the index
# This is important for time series analysis

time_series_weekly_vti_df = time_series_weekly_vti_df.sort_values('date').reset_index(drop=True)

# 6. Compute the Derived fields

    # A. Daily Price Movement
time_series_weekly_vti_df['daily_change'] = (
    time_series_weekly_vti_df['close'] - time_series_weekly_vti_df['open']
)

    # B. Intraday Return %
time_series_weekly_vti_df['intraday_return_pct'] = (
    (time_series_weekly_vti_df['close'] - time_series_weekly_vti_df['open'])
    / time_series_weekly_vti_df['open']
) * 100

    # C. Day-over-day return
time_series_weekly_vti_df['daily_return_pct'] = (
    time_series_weekly_vti_df['close']
    .pct_change()
) * 100

    # D. Cumulative return %
    # iloc[0] refers to the initial close
time_series_weekly_vti_df["cumulative_return_pct"] = (
    (time_series_weekly_vti_df["close"] / time_series_weekly_vti_df["close"].iloc[0]) - 1
) * 100

    # E. Intraday Volatility
time_series_weekly_vti_df['intraday_volatility_abs'] = (
    time_series_weekly_vti_df['high'] - time_series_weekly_vti_df['low']
)

    # F. Volatility Ratio
time_series_weekly_vti_df['volatility_ratio'] = (
    time_series_weekly_vti_df['intraday_volatility_abs']
    / time_series_weekly_vti_df['close']
)

    # G. Intraday Volatility %
time_series_weekly_vti_df['intraday_volatility_pct'] = (
    (
        time_series_weekly_vti_df['high']
        - time_series_weekly_vti_df['low']
    )
    / time_series_weekly_vti_df['open']
) * 100

    # H. Volume Change %
time_series_weekly_vti_df['volume_change_pct'] = (
    time_series_weekly_vti_df['volume']
    .pct_change()
) * 100

    # I. 7-Day Moving Average
time_series_weekly_vti_df['moving_avg_7d'] = (
    time_series_weekly_vti_df['close']
    .rolling(7)
    .mean()
)

    # J. 30-Day Moving Average
time_series_weekly_vti_df['moving_avg_30d'] = (
    time_series_weekly_vti_df['close']
    .rolling(30)
    .mean()
)

# 7. Add Symbol and Time Zone from the Metadata

time_series_weekly_vti_df['symbol'] = time_series_weekly_vti['Meta Data']['2. Symbol']
time_series_weekly_vti_df['dt_last_refreshed'] = time_series_weekly_vti['Meta Data']['3. Last Refreshed']
time_series_weekly_vti_df['timezone'] = time_series_weekly_vti['Meta Data']['4. Time Zone']

# 8. Convert data types from string to date
time_series_weekly_vti_df['dt_last_refreshed'] = pd.to_datetime(time_series_weekly_vti_df['dt_last_refreshed'])

# 8. Round numeric columns to 2 decimal places
numeric_columns = time_series_weekly_vti_df.select_dtypes(include='number').columns

time_series_weekly_vti_df[numeric_columns] = (
    time_series_weekly_vti_df[numeric_columns]
    .round(2)
)

# 9. View the data
time_series_weekly_vti_df

,date,open,high,low,close,volume,daily_change,intraday_return_pct,daily_return_pct,cumulative_return_pct,intraday_volatility_abs,volatility_ratio,intraday_volatility_pct,volume_change_pct,moving_avg_7d,moving_avg_30d,symbol,dt_last_refreshed,timezone
0,2001-06-08,116.10,118.00,115.30,116.40,2265500,0.30,0.26,NaN,0.00,2.70,0.02,2.33,NaN,NaN,NaN,VTI,2026-05-08,US/Eastern
1,2001-06-15,116.10,116.10,110.40,111.30,853500,-4.80,-4.13,-4.38,-4.38,5.70,0.05,4.91,-62.33,NaN,NaN,VTI,2026-05-08,US/Eastern
2,2001-06-22,111.60,113.50,110.50,112.30,1478600,0.70,0.63,0.90,-3.52,3.00,0.03,2.69,73.24,NaN,NaN,VTI,2026-05-08,US/Eastern
3,2001-06-29,112.20,113.60,110.10,113.00,431700,0.80,0.71,0.62,-2.92,3.50,0.03,3.12,-70.80,NaN,NaN,VTI,2026-05-08,US/Eastern
4,2001-07-06,112.60,113.90,109.00,109.00,875700,-3.60,-3.20,-3.54,-6.36,4.90,0.04,4.35,102.85,NaN,NaN,VTI,2026-05-08,US/Eastern
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296,2026-04-10,323.95,336.55,321.46,335.05,19371807,11.10,3.43,3.49,187.84,15.09,0.05,4.66,-13.78,326.94,332.71,VTI,2026-05-08,US/Eastern
1297,2026-04-17,333.94,351.74,333.64,350.53,18482054,16.59,4.97,4.62,201.14,18.11,0.05,5.42,-4.59,328.62,333.45,VTI,2026-05-08,US/Eastern
1298,2026-04-24,349.90,352.37,346.55,352.05,17522660,2.15,0.61,0.43,202.45,5.82,0.02,1.66,-5.19,331.56,334.28,VTI,2026-05-08,US/Eastern
1299,2026-05-01,352.00,357.15,348.67,355.29,15103707,3.29,0.93,0.92,205.23,8.48,0.02,2.41,-13.80,335.73,335.12,VTI,2026-05-08,US/Eastern


In [103]:
# 10. Check info
time_series_weekly_vti_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     1301 non-null   datetime64[us]
 1   open                     1301 non-null   float64       
 2   high                     1301 non-null   float64       
 3   low                      1301 non-null   float64       
 4   close                    1301 non-null   float64       
 5   volume                   1301 non-null   int64         
 6   daily_change             1301 non-null   float64       
 7   intraday_return_pct      1301 non-null   float64       
 8   daily_return_pct         1300 non-null   float64       
 9   cumulative_return_pct    1301 non-null   float64       
 10  intraday_volatility_abs  1301 non-null   float64       
 11  volatility_ratio         1301 non-null   float64       
 12  intraday_volatility_pct  1301 non-null   floa

In [104]:
# 11. Re-order columns

time_series_weekly_vti_df = time_series_weekly_vti_df[
    [
        'date',
        'symbol',
        'timezone',
        'open',
        'high',
        'low',
        'close',
        'volume',
        'daily_change',
        'intraday_return_pct',
        'daily_return_pct',
        'cumulative_return_pct',
        'intraday_volatility_abs',
        'volatility_ratio',
        'intraday_volatility_pct',
        'volume_change_pct',
        'moving_avg_7d',
        'moving_avg_30d'
    ]
]

time_series_weekly_vti_df

,date,symbol,timezone,open,high,low,close,volume,daily_change,intraday_return_pct,daily_return_pct,cumulative_return_pct,intraday_volatility_abs,volatility_ratio,intraday_volatility_pct,volume_change_pct,moving_avg_7d,moving_avg_30d
0,2001-06-08,VTI,US/Eastern,116.10,118.00,115.30,116.40,2265500,0.30,0.26,NaN,0.00,2.70,0.02,2.33,NaN,NaN,NaN
1,2001-06-15,VTI,US/Eastern,116.10,116.10,110.40,111.30,853500,-4.80,-4.13,-4.38,-4.38,5.70,0.05,4.91,-62.33,NaN,NaN
2,2001-06-22,VTI,US/Eastern,111.60,113.50,110.50,112.30,1478600,0.70,0.63,0.90,-3.52,3.00,0.03,2.69,73.24,NaN,NaN
3,2001-06-29,VTI,US/Eastern,112.20,113.60,110.10,113.00,431700,0.80,0.71,0.62,-2.92,3.50,0.03,3.12,-70.80,NaN,NaN
4,2001-07-06,VTI,US/Eastern,112.60,113.90,109.00,109.00,875700,-3.60,-3.20,-3.54,-6.36,4.90,0.04,4.35,102.85,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296,2026-04-10,VTI,US/Eastern,323.95,336.55,321.46,335.05,19371807,11.10,3.43,3.49,187.84,15.09,0.05,4.66,-13.78,326.94,332.71
1297,2026-04-17,VTI,US/Eastern,333.94,351.74,333.64,350.53,18482054,16.59,4.97,4.62,201.14,18.11,0.05,5.42,-4.59,328.62,333.45
1298,2026-04-24,VTI,US/Eastern,349.90,352.37,346.55,352.05,17522660,2.15,0.61,0.43,202.45,5.82,0.02,1.66,-5.19,331.56,334.28
1299,2026-05-01,VTI,US/Eastern,352.00,357.15,348.67,355.29,15103707,3.29,0.93,0.92,205.23,8.48,0.02,2.41,-13.80,335.73,335.12


In [105]:
# 12. Export to CSV and save inside output folder
time_series_weekly_vti_df.to_csv(
    output_folder / "time_series_weekly_vti_df.csv",
    index=False
)

In [106]:
# 13. Merge the VXX and VTI dataframes into 1
# Advantages when doing Data Visualization in Tableau
# A combined dataset allows filtering by symbol. line comparisons, dual-axis charts, easier dashboards and cleaner storytelling

combined_market_df = pd.concat(
    [
        time_series_weekly_vti_df,
        time_series_weekly_vxx_df
    ],
    ignore_index=True
)

combined_market_df

,date,symbol,timezone,open,high,low,close,volume,daily_change,intraday_return_pct,daily_return_pct,cumulative_return_pct,intraday_volatility_abs,volatility_ratio,intraday_volatility_pct,volume_change_pct,moving_avg_7d,moving_avg_30d,dt_last_refreshed
0,2001-06-08,VTI,US/Eastern,116.10,118.00,115.30,116.40,2265500,0.30,0.26,NaN,0.00,2.70,0.02,2.33,NaN,NaN,NaN,NaT
1,2001-06-15,VTI,US/Eastern,116.10,116.10,110.40,111.30,853500,-4.80,-4.13,-4.38,-4.38,5.70,0.05,4.91,-62.33,NaN,NaN,NaT
2,2001-06-22,VTI,US/Eastern,111.60,113.50,110.50,112.30,1478600,0.70,0.63,0.90,-3.52,3.00,0.03,2.69,73.24,NaN,NaN,NaT
3,2001-06-29,VTI,US/Eastern,112.20,113.60,110.10,113.00,431700,0.80,0.71,0.62,-2.92,3.50,0.03,3.12,-70.80,NaN,NaN,NaT
4,2001-07-06,VTI,US/Eastern,112.60,113.90,109.00,109.00,875700,-3.60,-3.20,-3.54,-6.36,4.90,0.04,4.35,102.85,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2197,2026-04-10,VXX,US/Eastern,34.80,36.33,30.10,30.77,49587252,-4.03,-11.58,-11.33,-68.51,6.24,0.20,17.92,-20.55,34.36,31.76,2026-05-08
2198,2026-04-17,VXX,US/Eastern,31.16,31.30,28.37,28.98,35726903,-2.18,-7.00,-5.82,-70.34,2.93,0.10,9.40,-27.95,34.37,31.62,2026-05-08
2199,2026-04-24,VXX,US/Eastern,29.29,30.94,29.07,29.79,39941119,0.50,1.71,2.80,-69.51,1.88,0.06,6.40,11.80,33.53,31.52,2026-05-08
2200,2026-05-01,VXX,US/Eastern,29.48,29.63,27.77,28.40,29256666,-1.08,-3.66,-4.67,-70.93,1.86,0.07,6.31,-26.75,32.57,31.35,2026-05-08


In [107]:
# 14. Export to CSV and save inside output folder

combined_market_df.to_csv(
    output_folder / "combined_market_data.csv",
    index=False
)

# Part 2: Cryptocurrencies

Your stakeholders are becoming aware of the rise in cryptocurrencies, and would like to understand the recent growth of this market. Your task is to use the Alpha Vantage API to extract historical data on cryptocurrency market performance, and tell a story about their growth using visuals created in a BI tool of your choice (Tableau/Power BI).

**1. Find the correct API endpoints to retrieve historical data on cryptocurrency prices over time. Daily should be a sufficient level of granularity for your purposes.**

Use any specific cryptocurrency you wish (e.g. Bitcoin) against the US Dollar.

Use the [Documentation](https://www.alphavantage.co/documentation/) to help you.

In [68]:
# STEP 1. DEFINE URL AND PARAMETERS

# API endpoint
base_url = 'https://www.alphavantage.co/query'

# Parameters
params = {
    'function': 'DIGITAL_CURRENCY_DAILY',    # returns the daily historical time series for a cryptocurrency traded on a specific market refreshed daily at midnight (UTC)
    'symbol': 'BTC',             # the cryptocurrency
    'market': 'USD',             # the exchange market
    'datatype': 'json',          # returns the daily time series in JSON format
    'apikey': ALPHAVANTAGE_API_KEY
}

**2. Use Python to read the data as JSON**

In [69]:
# STEP 2. API REQUEST + ERROR HANDLING

# 1. Calls the API via requests and safely handles possible errors

try:
    # Send GET request to API with headers and parameters
    response = requests.get(base_url, params=params)

    # Pause briefly to avoid hitting API rate limits
    time.sleep(1)

    # Raise an error if the response status is not successful
    response.raise_for_status()

    # Parse JSON response and convert to Python dictionary
    digital_currency_daily_btc_usd = response.json()

# 2. Handles HTTP errors 
# Catches API-related errors (e.g., 401, 404, 408, 429)

except requests.exceptions.HTTPError as e:
    print("HTTP error:", response.status_code, response.text)
    digital_currency_daily_btc_usd = [] # reset to empty dataset

# 3. Handles other errors 
# Catches unexpected errors (e.g., connection issues, parsing errors)

except Exception as e:
    print("Other error:", e)
    digital_currency_daily_btc_usd = [] # reset to empty dataset

In [70]:
digital_currency_daily_btc_usd

{'Meta Data': {'1. Information': 'Daily Prices and Volumes for Digital Currency',
  '2. Digital Currency Code': 'BTC',
  '3. Digital Currency Name': 'Bitcoin',
  '4. Market Code': 'USD',
  '5. Market Name': 'United States Dollar',
  '6. Last Refreshed': '2026-05-11 00:00:00',
  '7. Time Zone': 'UTC'},
 'Time Series (Digital Currency Daily)': {'2026-05-11': {'1. open': '82199.99000000',
   '2. high': '82379.00000000',
   '3. low': '81377.20000000',
   '4. close': '81789.99000000',
   '5. volume': '252.44843487'},
  '2026-05-10': {'1. open': '80655.97000000',
   '2. high': '82450.00000000',
   '3. low': '80273.86000000',
   '4. close': '82199.99000000',
   '5. volume': '3971.08807704'},
  '2026-05-09': {'1. open': '80188.82000000',
   '2. high': '81065.32000000',
   '3. low': '80115.33000000',
   '4. close': '80655.96000000',
   '5. volume': '2056.58900174'},
  '2026-05-08': {'1. open': '80005.38000000',
   '2. high': '80487.39000000',
   '3. low': '79520.44000000',
   '4. close': '80188

**3. Identify the key which holds the data itself and export it as a csv.**

In [ ]:
# STEP 3. CONVERT DATA IN JSON TO PANDAS DATAFRAME + DATA WRANGLING + COMPUTE DERIVED FIELDS + EXPORT TO A CSV FILE

# 1. Extract the 'Time Series (Digital Currency Daily)' portion and convert to a DataFrame
digital_currency_daily_btc_usd_df = pd.DataFrame.from_dict(
    digital_currency_daily_btc_usd['Time Series (Digital Currency Daily)'],
    orient='index'
) 

# 2. Reset index so date become a column instead of an index
digital_currency_daily_btc_usd_df.reset_index(inplace=True)

# 3. Rename columns
digital_currency_daily_btc_usd_df.columns = [
    'date',
    'open',
    'high',
    'low',
    'close',
    'volume'
]

# 4. Convert data types from string to date and float
digital_currency_daily_btc_usd_df['date'] = pd.to_datetime(digital_currency_daily_btc_usd_df['date'])

numeric_columns = ['open', 'high', 'low', 'close', 'volume']

digital_currency_daily_btc_usd_df[numeric_columns] = (
    digital_currency_daily_btc_usd_df[numeric_columns].astype(float)
)

# 5. Sort date by ascending order and resets the order of the index
# This is important for time series analysis

digital_currency_daily_btc_usd_df = digital_currency_daily_btc_usd_df.sort_values('date').reset_index(drop=True)

# 6. Compute the Derived fields

    # A. Daily Price Movement
digital_currency_daily_btc_usd_df['daily_change'] = (
    digital_currency_daily_btc_usd_df['close'] - digital_currency_daily_btc_usd_df['open']
)

    # B. Intraday Return %
digital_currency_daily_btc_usd_df['intraday_return_pct'] = (
    (digital_currency_daily_btc_usd_df['close'] - digital_currency_daily_btc_usd_df['open'])
    / digital_currency_daily_btc_usd_df['open']
) * 100

    # C. Day-over-day return
digital_currency_daily_btc_usd_df['daily_return_pct'] = (
    digital_currency_daily_btc_usd_df['close']
    .pct_change()
) * 100

    # D. Cumulative return %
    # iloc[0] refers to the initial close
digital_currency_daily_btc_usd_df["cumulative_return_pct"] = (
    (digital_currency_daily_btc_usd_df["close"] / digital_currency_daily_btc_usd_df["close"].iloc[0]) - 1
) * 100

    # E. Intraday Volatility
digital_currency_daily_btc_usd_df['intraday_volatility_abs'] = (
    digital_currency_daily_btc_usd_df['high'] - digital_currency_daily_btc_usd_df['low']
)

    # F. Intraday Volatility %
digital_currency_daily_btc_usd_df['intraday_volatility_pct'] = (
    (
        digital_currency_daily_btc_usd_df['high']
        - digital_currency_daily_btc_usd_df['low']
    )
    / digital_currency_daily_btc_usd_df['open']
) * 100

    # G. 7-Day Moving Average
digital_currency_daily_btc_usd_df['moving_avg_7d'] = (
    digital_currency_daily_btc_usd_df['close']
    .rolling(7)
    .mean()
)

    # H. 30-Day Moving Average
digital_currency_daily_btc_usd_df['moving_avg_30d'] = (
    digital_currency_daily_btc_usd_df['close']
    .rolling(30)
    .mean()
)

# 7. Add Symbol and Time Zone from the Metadata

digital_currency_daily_btc_usd_df['digital_ccy_code'] = digital_currency_daily_btc_usd['Meta Data']['2. Digital Currency Code']
digital_currency_daily_btc_usd_df['market_code'] = digital_currency_daily_btc_usd['Meta Data']['4. Market Code']
digital_currency_daily_btc_usd_df['dt_last_refreshed'] = digital_currency_daily_btc_usd['Meta Data']['6. Last Refreshed']
digital_currency_daily_btc_usd_df['time_zone'] = digital_currency_daily_btc_usd['Meta Data']['7. Time Zone']

# 8. Convert data types from string to date
digital_currency_daily_btc_usd_df['dt_last_refreshed'] = pd.to_datetime(digital_currency_daily_btc_usd_df['dt_last_refreshed'])

# 9. Round numeric columns to 2 decimal places
numeric_columns = digital_currency_daily_btc_usd_df.select_dtypes(include='number').columns

digital_currency_daily_btc_usd_df[numeric_columns] = (
    digital_currency_daily_btc_usd_df[numeric_columns]
    .round(2)
)

# 9. View the data
digital_currency_daily_btc_usd_df

,date,open,high,low,close,volume,daily_change,intraday_return_pct,daily_return_pct,cumulative_return_pct,intraday_volatility_abs,intraday_volatility_pct,moving_avg_7d,moving_avg_30d,digital_ccy_code,market_code,dt_last_refreshed,time_zone
0,2010-07-17,0.05,0.05,0.05,0.05,0.00,0.00,0.00,NaN,0.0,0.00,0.00,NaN,NaN,BTC,USD,2026-05-11,UTC
1,2010-07-18,0.09,0.09,0.09,0.09,0.00,0.00,0.00,71.60,71.6,0.00,0.00,NaN,NaN,BTC,USD,2026-05-11,UTC
2,2010-07-19,0.08,0.08,0.08,0.08,0.00,0.00,0.00,-5.83,61.6,0.00,0.00,NaN,NaN,BTC,USD,2026-05-11,UTC
3,2010-07-20,0.07,0.07,0.07,0.07,0.00,0.00,0.00,-7.55,49.4,0.00,0.00,NaN,NaN,BTC,USD,2026-05-11,UTC
4,2010-07-21,0.08,0.08,0.08,0.08,0.00,0.00,0.00,6.02,58.4,0.00,0.00,NaN,NaN,BTC,USD,2026-05-11,UTC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5773,2026-05-07,81438.11,81709.17,79456.00,80005.38,7038.15,-1432.73,-1.76,-1.76,160010660.0,2253.17,2.77,79668.40,76363.84,BTC,USD,2026-05-11,UTC
5774,2026-05-08,80005.38,80487.39,79520.44,80188.82,3086.74,183.44,0.23,0.23,160377540.0,966.95,1.21,79947.66,76667.27,BTC,USD,2026-05-11,UTC
5775,2026-05-09,80188.82,81065.32,80115.33,80655.96,2056.59,467.14,0.58,0.58,161311820.0,949.99,1.18,80229.61,76962.53,BTC,USD,2026-05-11,UTC
5776,2026-05-10,80655.97,82450.00,80273.86,82199.99,3971.09,1544.02,1.91,1.91,164399880.0,2176.14,2.70,80749.76,77269.27,BTC,USD,2026-05-11,UTC


In [72]:
digital_currency_daily_btc_usd_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5778 entries, 0 to 5777
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     5778 non-null   datetime64[us]
 1   open                     5778 non-null   float64       
 2   high                     5778 non-null   float64       
 3   low                      5778 non-null   float64       
 4   close                    5778 non-null   float64       
 5   volume                   5778 non-null   float64       
 6   daily_change             5778 non-null   float64       
 7   intraday_return_pct      5778 non-null   float64       
 8   daily_return_pct         5777 non-null   float64       
 9   cumulative_return_pct    5778 non-null   float64       
 10  intraday_volatility_abs  5778 non-null   float64       
 11  intraday_volatility_pct  5778 non-null   float64       
 12  moving_avg_7d            5772 non-null   floa

In [73]:
# 10. Re-order columns

digital_currency_daily_btc_usd_df = digital_currency_daily_btc_usd_df[
    [
        'date',
        'digital_ccy_code',
        'market_code',
        'dt_last_refreshed',
        'time_zone',
        'open',
        'high',
        'low',
        'close',
        'volume',
        'daily_change',
        'intraday_return_pct',
        'daily_return_pct',
        'cumulative_return_pct',
        'intraday_volatility_abs',
        'intraday_volatility_pct',
        'moving_avg_7d',
        'moving_avg_30d'
    ]
]

digital_currency_daily_btc_usd_df

,date,digital_ccy_code,market_code,dt_last_refreshed,time_zone,open,high,low,close,volume,daily_change,intraday_return_pct,daily_return_pct,cumulative_return_pct,intraday_volatility_abs,intraday_volatility_pct,moving_avg_7d,moving_avg_30d
0,2010-07-17,BTC,USD,2026-05-11,UTC,0.05,0.05,0.05,0.05,0.00,0.00,0.00,NaN,0.0,0.00,0.00,NaN,NaN
1,2010-07-18,BTC,USD,2026-05-11,UTC,0.09,0.09,0.09,0.09,0.00,0.00,0.00,71.60,71.6,0.00,0.00,NaN,NaN
2,2010-07-19,BTC,USD,2026-05-11,UTC,0.08,0.08,0.08,0.08,0.00,0.00,0.00,-5.83,61.6,0.00,0.00,NaN,NaN
3,2010-07-20,BTC,USD,2026-05-11,UTC,0.07,0.07,0.07,0.07,0.00,0.00,0.00,-7.55,49.4,0.00,0.00,NaN,NaN
4,2010-07-21,BTC,USD,2026-05-11,UTC,0.08,0.08,0.08,0.08,0.00,0.00,0.00,6.02,58.4,0.00,0.00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5773,2026-05-07,BTC,USD,2026-05-11,UTC,81438.11,81709.17,79456.00,80005.38,7038.15,-1432.73,-1.76,-1.76,160010660.0,2253.17,2.77,79668.40,76363.84
5774,2026-05-08,BTC,USD,2026-05-11,UTC,80005.38,80487.39,79520.44,80188.82,3086.74,183.44,0.23,0.23,160377540.0,966.95,1.21,79947.66,76667.27
5775,2026-05-09,BTC,USD,2026-05-11,UTC,80188.82,81065.32,80115.33,80655.96,2056.59,467.14,0.58,0.58,161311820.0,949.99,1.18,80229.61,76962.53
5776,2026-05-10,BTC,USD,2026-05-11,UTC,80655.97,82450.00,80273.86,82199.99,3971.09,1544.02,1.91,1.91,164399880.0,2176.14,2.70,80749.76,77269.27


In [74]:
# 11. Export CSV and Save inside folder
digital_currency_daily_btc_usd_df.to_csv(
    output_folder / "digital_currency_daily_btc_usd_df.csv",
    index=False
)